In [1]:
import sys

import pandas as pd

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts

from dice4el.scenario.scenario_handler import ScenarioHandler
from dice4el.scenario.scenario_model import ScenarioLSTM, train_ScenarioLSTM, validate_ScenarioLSTM

### --- Load Dataset ---

In [2]:
set_seed(seed=42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
df = pd.read_excel(
    "../../../data/bpic20_Rfp.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Activity": "string",
        "case:OrganizationalEntity": "string",
        "case:RequestedAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [6]:
df.head(20)

,case:concept:name,time:timestamp,case:Activity,case:OrganizationalEntity,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,request for payment 147529,2017-02-14 15:34:34,UNKNOWN,organizational unit 65458,137.526306,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,request for payment 147529,2017-02-14 15:34:43,UNKNOWN,organizational unit 65458,137.526306,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,9.0
2,request for payment 147529,2017-02-15 14:48:02,UNKNOWN,organizational unit 65458,137.526306,Request Payment,SYSTEM,UNDEFINED,83599.0
3,request for payment 147529,2017-02-20 17:32:08,UNKNOWN,organizational unit 65458,137.526306,Payment Handled,SYSTEM,UNDEFINED,441846.0
4,request for payment 147534,2017-03-02 15:55:43,UNKNOWN,organizational unit 65463,59.567024,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
5,request for payment 147534,2017-03-02 15:58:27,UNKNOWN,organizational unit 65463,59.567024,Request For Payment APPROVED by PRE_APPROVER,STAFF MEMBER,PRE_APPROVER,164.0
6,request for payment 147534,2017-03-02 16:07:38,UNKNOWN,organizational unit 65463,59.567024,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,551.0
7,request for payment 147534,2017-03-06 13:57:31,UNKNOWN,organizational unit 65463,59.567024,Request Payment,SYSTEM,UNDEFINED,337793.0
8,request for payment 147534,2017-03-13 17:31:05,UNKNOWN,organizational unit 65463,59.567024,Payment Handled,SYSTEM,UNDEFINED,617614.0
9,request for payment 147539,2017-03-06 14:40:07,UNKNOWN,organizational unit 65458,47.927757,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0


In [7]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Activity', 'case:OrganizationalEntity', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [3.00, 325445.40]                        57230.0000 quantile_derived    
case:RequestedAmount           continuous     case     yes    [10.54, 665.70]                          74.7009    quantile_derived    
org:resource                   categorical    event    yes    ['STAFF MEMBER', 'SYSTEM']               N/A        data_derived        
or

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

### --- Scenario Model ---

In [9]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [10]:
scenario_df = scenario_handler.generate_scenario_df(
    df=df,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
    n_scenarios_per_length=3
)

In [11]:
scenario_df.head()

,case:concept:name,time_index,fake,case:Activity,case:OrganizationalEntity,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,f_17359,0,True,activity 181118,organizational unit 65472,357.145706,Request For Payment SUBMITTED by EMPLOYEE,SYSTEM,UNDEFINED,11.858111
1,f_17359,1,True,activity 181118,organizational unit 65472,357.145706,Request For Payment REJECTED by MISSING,SYSTEM,SUPERVISOR,7371.566440
2,f_17359,2,True,activity 181118,organizational unit 65472,357.145706,Request For Payment FINAL_APPROVED by BUDGET O...,STAFF MEMBER,BUDGET OWNER,916.442194
3,f_17359,3,True,activity 181118,organizational unit 65472,357.145706,Request Payment,STAFF MEMBER,EMPLOYEE,1132.464748
4,f_17359,4,True,activity 181118,organizational unit 65472,357.145706,Payment Handled,SYSTEM,MISSING,3.940568


In [12]:
case_ids = scenario_df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = scenario_df[scenario_df["case:concept:name"].isin(train_cases)].copy()
val_df   = scenario_df[scenario_df["case:concept:name"].isin(val_cases)].copy()

In [13]:
train_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=train_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [14]:
val_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=val_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [15]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [16]:
criterion = torch.nn.BCEWithLogitsLoss()

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Rfp-scenario_model_output.txt")

Epoch 020/100 | Train Loss: 0.0036 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.0029 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.0024 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.0020 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.0017 | LR: 1.00e-06
Time taken for scenario model (training): 2026.093513 seconds
Time taken for scenario model (validation): 1.420905 seconds
Val loss: {'loss': 0.011630285585408203, 'accuracy': 0.998471363543719, 'f1_macro': 0.9979217187898154, 'f1_weighted': 0.9984721259811212}


In [18]:
embedding_metadata = scenario_handler.get_scenario_embedding_metadata()

scenario_model = ScenarioLSTM(
    categorical_info=embedding_metadata["categorical_info"],
    n_continuous=embedding_metadata["n_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ScenarioLSTM(
    model=scenario_model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

scenario_model.save()

In [19]:
# scenario_model = ScenarioLSTM.load()

In [20]:
val_loss = validate_ScenarioLSTM(
    model=scenario_model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [21]:
sys.stdout = original_stdout
log_file.close()